# Results analysis
Run `make full` (or `make quick`) first. This notebook reads `results/*/seed*/metrics.json`.

In [ ]:
import sys; sys.path.insert(0, '../src')
import pandas as pd, matplotlib.pyplot as plt
from efficientlm.summarize import load_runs, aggregate, to_markdown
RESULTS = '../results'
runs = load_runs(RESULTS)
print(len(runs), 'runs found')

## 1. Per-run table

In [ ]:
df = pd.DataFrame([{k: r[k] for k in ['method','seed','test_acc','test_f1','trainable_params','model_size_mb','latency_ms','train_time_s']} for r in runs])
df.sort_values(['method','seed'])

## 2. Mean ± std over seeds

In [ ]:
from IPython.display import Markdown
Markdown(to_markdown(aggregate(runs)))

## 3. Learning curves
Do the LoRA models converge as fast as full fine-tuning?

In [ ]:
plt.figure(figsize=(6,4))
for r in runs:
    if r['seed'] == 0:
        plt.plot([h['epoch'] for h in r['history']], [h['dev_acc'] for h in r['history']], marker='o', label=r['method'])
plt.xlabel('epoch'); plt.ylabel('dev accuracy'); plt.legend(); plt.grid(alpha=.3); plt.show()

## 4. LoRA rank ablation
Does a bigger rank help? Compare `lora_r4`, `lora_r8`, `lora_r16` (differences smaller than the std are noise).

In [ ]:
lora = df[df.method.str.startswith('lora_r')].groupby('method').test_acc.agg(['mean','std','count'])
lora

## 5. Does distillation help the small model?
`student_kd` vs `student_ce` use the **same** architecture, data and epochs, only the loss differs.

In [ ]:
pair = df[df.method.isin(['student_ce','student_kd'])].pivot(index='seed', columns='method', values='test_acc')
pair['gain'] = pair['student_kd'] - pair['student_ce']
print(pair)
print('mean gain: %.4f' % pair['gain'].mean())

## 6. Your conclusions
Write 5-6 sentences: what worked, what did not, whether differences exceed the seed-to-seed noise, and what you would try next.